In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
df = pd.read_csv("Forest_Area.csv")
forest_area_cols = [col for col in df.columns if 'Forest Area' in col and 'Proportion' not in col]
target_col = 'Deforestation, 2015-2020'
other_numeric_cols = ['Total Land Area, 2020'] 
cols_to_clean = forest_area_cols + [target_col] + other_numeric_cols
for col in cols_to_clean:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df = df[df['Country and Area'] != 'WORLD'].reset_index(drop=True)
features = [
    'Country and Area',
    'Forest Area, 1990',
    'Forest Area, 2000',
    'Forest Area, 2010',
    'Forest Area, 2015',
    'Total Land Area, 2020',
    'Forest Area as a Proportion of Total Land Area, 2020'
]
model_df = df[features + [target_col]].copy()
model_df.dropna(inplace=True)
Y = model_df[target_col]
X = model_df.drop(target_col, axis=1)
X_encoded = pd.get_dummies(X, columns=['Country and Area'], drop_first=True)
X_train, X_test, Y_train, Y_test = train_test_split(
    X_encoded, Y, test_size=0.2, random_state=42
)
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_regressor.fit(X_train, Y_train)
Y_pred = rf_regressor.predict(X_test)
rmse = np.sqrt(mean_squared_error(Y_test, Y_pred))
r2 = r2_score(Y_test, Y_pred)
feature_importances = pd.Series(rf_regressor.feature_importances_, index=X_encoded.columns)
top_10_features = feature_importances.nlargest(10)
print("\n--- Training Random Forest Regressor ---")
print("\n--- Model Performance Metrics ---")
print(f"R-squared (R2) Score: {r2:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print("\n--- Top 10 Feature Importances (Key Drivers) ---")
print(top_10_features.to_markdown(numalign="left", stralign="left"))


--- Training Random Forest Regressor ---

--- Model Performance Metrics ---
R-squared (R2) Score: 0.6660
Root Mean Squared Error (RMSE): 42.26

--- Top 10 Feature Importances (Key Drivers) ---
|                                                      | 0         |
|:-----------------------------------------------------|:----------|
| Country and Area_Brazil                              | 0.286091  |
| Forest Area, 1990                                    | 0.211149  |
| Country and Area_Indonesia                           | 0.0982329 |
| Country and Area_India                               | 0.0860853 |
| Forest Area, 2010                                    | 0.0709535 |
| Forest Area, 2000                                    | 0.0599762 |
| Forest Area, 2015                                    | 0.0511237 |
| Total Land Area, 2020                                | 0.0416769 |
| Country and Area_United Republic of Tanzania         | 0.0383755 |
| Forest Area as a Proportion of Total Land Are